In [8]:
import json
import zipfile
from pathlib import Path
import pandas as pd
import warnings


# Configuration
# ---------------------------------------------------------------------

zip_path = Path(
    "/home/devd/Downloads/"
    "stage1_scope-head_only_no_stage2-1ebe22aedfe6-scientific-record_v0.zip"
)

test_csv_basename = "predictions_test_best.csv"
unseen_csv_basename = "predictions_unseen_best.csv"

filename_column = "image_id"
species_column = "predicted_species"


# ---------------------------------------------------------------------
# Find and read the two prediction files
# ---------------------------------------------------------------------

with zipfile.ZipFile(zip_path) as archive:
    csv_files = [
        name
        for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]

    print("CSV files in archive:")
    for name in csv_files:
        print("  ", name)

    # Match by basename in case the files are inside a directory in the ZIP.
    files_by_basename = {
        Path(name).name: name
        for name in csv_files
    }

    required_files = {
        test_csv_basename,
        unseen_csv_basename,
    }

    missing_csvs = required_files - files_by_basename.keys()

    if missing_csvs:
        raise FileNotFoundError(
            f"Required prediction CSV files are missing: {sorted(missing_csvs)}"
        )

    with archive.open(files_by_basename[test_csv_basename]) as file:
        test_df = pd.read_csv(file)

    with archive.open(files_by_basename[unseen_csv_basename]) as file:
        unseen_df = pd.read_csv(file)


# ---------------------------------------------------------------------
# Validate and clean each split
# ---------------------------------------------------------------------

def prepare_predictions(
    frame: pd.DataFrame,
    split_name: str,
) -> pd.DataFrame:
    required_columns = {
        filename_column,
        species_column,
    }

    missing_columns = required_columns - set(frame.columns)

    if missing_columns:
        raise ValueError(
            f"{split_name} is missing columns: {sorted(missing_columns)}. "
            f"Available columns: {frame.columns.tolist()}"
        )

    result = frame[
        [filename_column, species_column]
    ].copy()

    # Pandas' string dtype preserves missing values as <NA>.
    result[filename_column] = (
        result[filename_column]
        .astype("string")
        .str.strip()
    )

    result[species_column] = (
        result[species_column]
        .astype("string")
        .str.strip()
    )

    missing_mask = (
        result[filename_column].isna()
        | result[species_column].isna()
        | result[filename_column].eq("")
        | result[species_column].eq("")
    )

    if missing_mask.any():
        print(f"\nInvalid rows in {split_name}:")
        print(result.loc[missing_mask].to_string(index=False))
        raise ValueError(
            f"{split_name} contains missing filenames or predictions."
        )

    # Ground-truth keys appear to use bare filenames such as 96566.jpg.
    result[filename_column] = result[filename_column].map(
        lambda value: Path(str(value)).name
    )

    invalid_filenames = result[
        ~result[filename_column].str.lower().str.endswith(".jpg")
    ]

    if not invalid_filenames.empty:
        print(f"\nInvalid filenames in {split_name}:")
        print(invalid_filenames.head(20).to_string(index=False))
        raise ValueError(
            f"{split_name} contains filenames that do not end in .jpg."
        )

    duplicated = result[
        result[filename_column].duplicated(keep=False)
    ]

    if not duplicated.empty:
        print(f"\nDuplicate filenames in {split_name}:")
        print(
            duplicated
            .sort_values(filename_column)
            .head(30)
            .to_string(index=False)
        )
        raise ValueError(
            f"{split_name} contains duplicate filenames."
        )

    return result


test_predictions = prepare_predictions(
    test_df,
    split_name="test",
)

unseen_predictions = prepare_predictions(
    unseen_df,
    split_name="unseen",
)


# ---------------------------------------------------------------------
# Combine both splits
# ---------------------------------------------------------------------

combined_df = pd.concat(
    [test_predictions, unseen_predictions],
    ignore_index=True,
)

cross_split_duplicates = combined_df[
    combined_df[filename_column].duplicated(keep=False)
]

if not cross_split_duplicates.empty:
    print("\nFilenames appearing in both splits:")
    print(
        cross_split_duplicates
        .sort_values(filename_column)
        .head(30)
        .to_string(index=False)
    )
    raise ValueError(
        "Duplicate filenames were found after combining the splits."
    )


# Explicit str() ensures ordinary Python strings are written to JSON.
predictions = {
    str(filename): str(species)
    for filename, species in zip(
        combined_df[filename_column],
        combined_df[species_column],
    )
}


# ---------------------------------------------------------------------
# Final validation
# ---------------------------------------------------------------------

assert len(predictions) == len(combined_df)
assert all(isinstance(key, str) for key in predictions)
assert all(isinstance(value, str) for value in predictions.values())
assert all(key.lower().endswith(".jpg") for key in predictions)
assert all(value.strip() for value in predictions.values())

print("\nPrediction counts")
print("-----------------")
print("Test:    ", len(test_predictions))
print("Unseen:  ", len(unseen_predictions))
print("Combined:", len(combined_df))
print("JSON keys:", len(predictions))

print("\nFirst five predictions:")
for item in list(predictions.items())[:5]:
    print(item)


# ---------------------------------------------------------------------
# Write prediction.json
# ---------------------------------------------------------------------

output_path = zip_path.parent / "prediction.json"

with output_path.open("w", encoding="utf-8") as file:
    json.dump(
        predictions,
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )


# ---------------------------------------------------------------------
# Create submission.zip
# ---------------------------------------------------------------------

submission_path = zip_path.parent / "submission.zip"
submission_path.unlink(missing_ok=True)

with zipfile.ZipFile(
    submission_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        output_path,
        arcname="prediction.json",
    )


CSV files in archive:
   classification_report_best_genus.csv
   classification_report_best_species.csv
   classification_report_genus.csv
   classification_report_last_genus.csv
   classification_report_last_species.csv
   classification_report_species.csv
   history.csv
   history_stage1.csv
   predictions_test_best.csv
   predictions_unseen_best.csv

Prediction counts
-----------------
Test:     20097
Unseen:   15568
Combined: 35665
JSON keys: 35665

First five predictions:
('230675.jpg', 'Calloplesiops altivelis')
('59785.jpg', 'Calloplesiops altivelis')
('288569.jpg', 'Calloplesiops altivelis')
('372471.jpg', 'Calloplesiops altivelis')
('255592.jpg', 'Arothron stellatus')


In [9]:
combined_df = pd.concat(
    [test_df, unseen_df],
    ignore_index=True,
)

predictions = dict(
    zip(
        combined_df["image_id"].astype(str),
        combined_df["predicted_species"].astype(str),
    )
)

output_path = zip_path.parent / "prediction.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

print("Test predictions:", len(test_df))
print("Unseen predictions:", len(unseen_df))
print("Combined predictions:", len(predictions))
print("Created:", output_path)

Test predictions: 20097
Unseen predictions: 15568
Combined predictions: 35665
Created: /home/devd/Downloads/prediction.json


In [10]:
submission_path = zip_path.parent / "submission.zip"

submission_path.unlink(missing_ok=True)

with zipfile.ZipFile(
    submission_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    z.write(output_path, arcname="prediction.json")

print("Created:", submission_path)

Created: /home/devd/Downloads/submission.zip
